# Atcoder Typical DP Contest

> "전형적인 DP 문제를 풀어보자"

- toc: true
- branch: master
- badges: true
- comments: true
- author: 한재수
- categories: [Algorithm]

`-` 대회 링크: https://atcoder.jp/contests/tdpc

`-` 현재 [EDPC](https://atcoder.jp/contests/dp)에서 풀지 못한 문제가 5개(U, V, W, X, Y) 남았는데 오래 걸릴 것 같아서 여기로 이사왔다 

`-` 총 20 문제가 있는데 시간 날 때마다 풀어보자 (일본어라 번역해야 돼). 이번엔 다 풀 수 있을까?

## A - Contest

`-` $N$개의 질문을 고려하자. 각 질문을 해결하여 점수를 얻거나 해결하지 않아 점수를 얻지 않을 수 있다. 이를 통해 최종 점수로 가능한 것들이 총 몇 개 있는지 세면 된다

`-` $\operatorname{dp}[i][s]$를 $i$번째 질문까지 고려했을 때 $s$점을 획득할 수 있으면 $\text{True}$, 아니면 $\text{False}$로 정의하자. 그럼 점화식은 다음과 같다

`-` $\operatorname{dp}[i][s] = \operatorname{dp}[i - 1][s] \lor \operatorname{dp}[i - 1][s - p_i]$. base case로 $\operatorname{dp}[0][0] = \text{True},\, \operatorname{dp}[i][p_i] = \text{True}$이며 정답은 $\sum \operatorname{dp}[N]$이다

`-` 문제 조건에서 $p_i \le 100$이므로 최대 획득 점수는 $100N$에 바운드된다. 따라서 전체 알고리즘의 시간 복잡도는 $O\left(100N^2\right)$이다

`-` 배낭 용량과 물건 가치는 고려하지 않고 만들 수 있는 배낭 무게의 가짓수만 관심있는 배낭 문제라고 생각할 수 있겠다~

In [7]:
#collapse
def solution():
    N = int(input())
    array = list(map(int, input().split()))
    max_score = 100 * N
    dp = [0] * (max_score + 1)
    dp[0] = 1 
    for i in range(N):
        p_i = array[i]
        for s in range(max_score, p_i, -1):
            dp[s] |= dp[s - p_i]
        dp[p_i] = 1
    answer = sum(dp)
    print(answer)


solution()

`-` 배열로 풀긴 했지만 이런 유형은 해시셋 딸깍 하는 게 더 쉽다

`-` 비슷한 문제로 [양팔저울](https://jungol.co.kr/problem/1352)이 있다

## B - Game

`-` 각자 번갈아가며 왼쪽 더미와 오른쪽 더미 중 하나를 골라 맨 위의 물건을 가져간다. 각자가 최선을 다했을 때 선공 플레이어가 가져간 물건의 총 가치를 구하면 된다

`-` 최종적으로 선공 플레이어가 가져간 물건의 총 가치와 후공 플레이어가 가져간 물건의 총 가치의 합은 전체 물건의 총 가치와 동일하다. 최선을 다하는 건 게임에서 승리하도록 행동한다는 것이고 이는 내가 가져간 물건의 총 가치가 상대가 가져간 물건의 총 가치보다 크게 만드는 것이다. 각 플레이어는 지금 점수(=가져간 물건의 총 가치)를 많이 획득하는 것에 신경을 쓰는 게 아니라 게임이 종료되는 시점에 점수 차이가 극대화되는 선택을 한다

`-` $\operatorname{dp}[i][j]$를 밑에서부터 왼쪽 더미의 물건 $i$개와 오른쪽 더미의 물건 $j$개만 고려한 게임에서 각자 최선을 다해 플레이 할 때 게임 종료시 $\text{본인의 점수 $-$ 상대방의 점수}$로 정의하자. 현재 플레이어는 왼쪽 더미에서 물건을 가져가거나 오른쪽 더미에서 물건을 가져갈 수 있다. 따라서 점화식은 다음과 같다

`-` $\operatorname{dp}[i][j] = \max(a_{A-i + 1} - \operatorname{dp}[i - 1][j],\, b_{B-j+1} - \operatorname{dp}[i][j - 1])$

`-` base case로 $\operatorname{dp}[0][0] = 0$이다. 게임이 종료된 시점에서 $\text{선공 플레이어의 점수 $-$ 후공 플레이어의 점수 = $\operatorname{dp}[A][B]$}$이다

`-` $\text{선공 플레이어의 점수 $+$ 후공 플레이어의 점수 $=$ 물건의 총 가치}$이므로 연립 방정식을 통해 정답을 얻을 수 있다. 전체 알고리즘의 시간 복잡도는 $O(AB)$이다

`-` 바텁업으로 구현할 때 남은 물건 개수를 기준으로 오름차순으로 계산해야 함에 주의하자 (물건이 $1$개 남았을 때 가능한 경우 계산 $\to$ 물건이 $2$개 남았을 때 가능한 경우 계산 $\to \cdots \to$ 물건이 $A+B$개 남았을 때 가능한 경우 계산)

In [51]:
#collapse
def solution():
    inf = 10**6
    A, B = map(int, input().split())
    a = list(map(int, input().split()))
    b = list(map(int, input().split()))
    dp = [[0] * (B + 1) for _ in range(A + 1)]
    for remainder in range(1, A + B + 1):
        for i in range(max(remainder - B, 0), min(remainder, A) + 1):
            j = remainder - i
            left_pick = a[A - i] - dp[i - 1][j] if i > 0 else -inf
            right_pick = b[B - j] - dp[i][j - 1] if j > 0 else -inf
            dp[i][j] = max(left_pick, right_pick)
    answer = (dp[A][B] + sum(a) + sum(b)) // 2
    print(answer)


solution()

`-` 이런 유형을 Minimax DP로 부른다

`-` 비슷한 문제로 [Deque](https://atcoder.jp/contests/dp/tasks/dp_l)이 있다

## 문제 푼 순서

|No.|문제|날짜|
|:-:|:-:|:-:|
|1  |A  |2026-10-02|
|2  |B  |2026-10-02|